In [1]:
# Saving country level OSDMA8 for all ensemble members

In [1]:
import xarray as xr
import numpy as np

In [2]:
# === Latitude weighting mean ===
def weighted_mean(da):
    weights = np.cos(np.deg2rad(da.lat))
    weights.name = "weights"
    new_da = da.weighted(weights).mean(("lon", "lat"))
    return new_da

In [3]:
# === Path config ===
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

country_mask = xr.open_dataarray(f"{MASK_DIR}GBD_Country_Masks_0.10_popgrid_newlabels.nc")

In [4]:
# === Path config ===
OSDMA8_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"
# SCENARIOS = ["ARISE", "SSP245"]
SCENARIOS = ["ARISE"]


# === Main loop ===
for scenario in SCENARIOS:
    ensembles = []
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        da = xr.open_dataarray(f"{OSDMA8_DIR}OSDMA8_BC_popgrid_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")

        countries = []
        # Loop over countries and take the OSDMA8 mean for each country
        for i in range(len(country_mask["country"])):
            mask = country_mask.isel(country=i)
            o3_country = weighted_mean(xr.where(mask == 1, da, np.nan))  # osdma8 of country
            countries.append(o3_country.drop_vars("country", errors='ignore'))
        osdma8_country = xr.concat(countries, dim=xr.DataArray(country_mask["country"], dims="country", name="country"))

        ensembles.append(osdma8_country)

    country_ens = xr.concat(ensembles, dim=xr.DataArray(np.arange(1, len(ensembles)+1), dims="ensemble", name="ensemble"))

    print(f"Saving country level OSDMA8 to {OSDMA8_DIR}")
    country_ens.to_netcdf(f"{OSDMA8_DIR}OSDMA8_BC_Country_mean_CESM2_{scenario}_{dates}.nc")

Processing ARISE, Ensemble 01
Processing ARISE, Ensemble 02
Processing ARISE, Ensemble 03
Processing ARISE, Ensemble 04
Processing ARISE, Ensemble 05
Processing ARISE, Ensemble 06
Processing ARISE, Ensemble 07
Processing ARISE, Ensemble 08
Processing ARISE, Ensemble 09
Processing ARISE, Ensemble 10
Saving country level OSDMA8 to /glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/


RuntimeError: NetCDF: HDF error